# 15 — Descobrir vídeos novos por canal
Consulta a playlist de uploads dos canais habilitados e cadastra somente IDs novos na fila existente.

In [ ]:
import json

from youtube_etl_genai.main import _get_api_key, _get_spark_session
from youtube_etl_genai.observability import TaskExecution, configure_job_logging
from youtube_etl_genai.pipeline import discover_channel_videos_step

TASK_KEY = "discover_channel_videos"
configure_job_logging()

for name, default in [
    ("catalog", "youtube_lakehouse"),
    ("new_video_priority", "100"),
    ("new_video_refresh_interval_hours", "24"),
    ("task_run_id", ""),
    ("secret_scope", "youtube_api_key"),
    ("secret_key", "api-key"),
]:
    dbutils.widgets.text(name, default)

spark = _get_spark_session()
catalog = dbutils.widgets.get("catalog")
with TaskExecution(
    spark=spark,
    catalog=catalog,
    task_key=TASK_KEY,
    task_run_id=dbutils.widgets.get("task_run_id") or None,
) as task_execution:
    result = discover_channel_videos_step(
        spark=spark,
        api_key=_get_api_key(
            spark,
            dbutils.widgets.get("secret_scope"),
            dbutils.widgets.get("secret_key"),
        ),
        new_video_priority=dbutils.widgets.get("new_video_priority"),
        new_video_refresh_interval_hours=dbutils.widgets.get(
            "new_video_refresh_interval_hours"
        ),
        catalog=catalog,
        api_cost_observer=task_execution.add_api_cost,
    )
    task_execution.complete_from_result(result)

print(json.dumps(result, sort_keys=True))
